# MedVis Exercise Sheet 2 — Task 4: Image Segmentation

Goal: Segment brain gray matter from slice 18 of dataset1 using **thresholding**, then improve the result using **Connected Component Analysis (CCA)**.

## Preparation

In [ ]:
!pip install scipy
!pip install pydicom
!pip install scikit-image

import numpy as np
import matplotlib.pyplot as plt
from pydicom import dcmread
from skimage.measure import label, regionprops, regionprops_table

## Step 1: Read Slice 18

In [ ]:
# Read slice 18
dcm_slice = dcmread('dataset1/brain_018.dcm')
img = np.array(dcm_slice.pixel_array)

# Display original
fig = plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Brain Slice 18 (Original)')
plt.axis('off')
plt.show()

## Step 2: Thresholding

We label every pixel with intensity between 200 and 400 as tissue (True = white) and everything else as background (False = black). This range targets gray matter based on its typical MRI signal intensity in this dataset.

In [ ]:
# Apply thresholding
threshold_low  = 200
threshold_high = 400
segmentation = (img >= threshold_low) & (img <= threshold_high)

# Display original vs thresholding result
fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.set_title('Original')
ax1.imshow(img, cmap='gray')
ax1.axis('off')

ax2 = fig.add_subplot(1, 2, 2)
ax2.set_title(f'Thresholding ({threshold_low}–{threshold_high})')
ax2.imshow(segmentation, cmap='gray')
ax2.axis('off')

plt.tight_layout()
plt.show()

print('Notice: many small unrelated regions are also segmented.')
print('This is because other tissues share similar intensity values to gray matter.')

## Step 3: Connected Component Analysis (Optional Improvement)

Thresholding alone picks up many unrelated structures. CCA labels each connected group of white pixels separately, then we keep only the largest — which is the gray matter.

In [ ]:
# Label connected components
label_im = label(segmentation)
label_im  # <- matrix in which each pixel gets a label

# Calculate statistics over all regions
regions = regionprops(label_im)

# Sort the regions in ascending order of size
lst_sorted = sorted(regions, key=lambda x: x.area)

# Take the data from the last index from the list (largest region)
graymatter = label_im == lst_sorted[-1].label

print(f'Total regions found: {len(regions)}')
print(f'Largest region area: {lst_sorted[-1].area} pixels')
print(f'2nd largest area:    {lst_sorted[-2].area} pixels')

In [ ]:
# Display all results side by side
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(1, 3, 1)
ax1.set_title('Original')
ax1.imshow(img, cmap='gray')
ax1.axis('off')

ax2 = fig.add_subplot(1, 3, 2)
ax2.set_title(f'Thresholding ({threshold_low}–{threshold_high})')
ax2.imshow(segmentation, cmap='gray')
ax2.axis('off')

ax3 = fig.add_subplot(1, 3, 3)
ax3.set_title('After CCA (Gray Matter)')
ax3.imshow(graymatter, cmap='gray')
ax3.axis('off')

plt.tight_layout()
plt.show()

## Discussion

### Why can't thresholding achieve a perfect segmentation?
Thresholding only asks *'is this pixel between value X and Y?'* with no knowledge of shape, location, or anatomical context. In MRI, different tissues share overlapping intensity ranges, so other structures with similar brightness also get segmented. The **partial volume effect** also causes boundary pixels to contain mixed tissue signals, placing them incorrectly inside the threshold range.

### How does CCA improve it?
CCA labels each connected group of white pixels and computes their area. Since gray matter forms one large connected region while noise fragments are tiny isolated islands, keeping only the largest region gives a much cleaner segmentation.